# 03 — Elastic Net Selection

**Thesis:** *Nowcasting and Indicator Selection in a Data-Rich Environment: An Application to German GDP Growth*

Part I selector: at each monthly origin the elastic net is estimated on **completed-quarter** aggregates. Training ends at the last finished quarter, so the three monthly origins in a quarter share the same set. Publication lags are left to the nowcasting models in `06_dfm_nowcasting.ipynb`.

The thesis comparison uses four signals: the capped elastic net, a block-balanced re-ranking of those coefficients (one series per category, *k* = 20; built in `scripts/pipelines/dfm/run_blockbalanced_benchmark.py`, not here), PLS+VIP, and XGBoost SHAP. A fixed-*k* = 30 Bai & Ng (2008) path is run in this notebook as a diagnostic. It is **not** the block-balanced set.

## Specification (matches the thesis appendix)

| Choice | Setting |
|---|---|
| Penalty path | 40 log-spaced α on [10⁻³, 10]; mixing ρ ∈ {0.1, 0.5, 0.9, 1.0} (`DEFAULT_ALPHAS`, `DEFAULT_L1_RATIOS`) |
| CV | five-fold time-ordered splits |
| Pre-filter | keep series with \|t̂_j\| ≥ 1.65; bypassed if fewer than seven survive |
| Imputation | iterative (MICE), 10 iterations, seed 42; fitted once per training window |
| COVID weights | 2020Q2–2021Q1 receive weight 0.25 (Lenza & Primiceri 2022) |
| Cap | if more than 60 coefficients are non-zero, keep the 60 largest in absolute value |

Across the 180 origins the capped sets range from 12 to 60 series (mean 51.5, median 58.5). From 2020Q2 the cap binds at every refit, so later composition changes are substitutions under a fixed budget.

## Outputs (`outputs/indicator_selection/`)

| File | Content |
|---|---|
| `gdp_target.csv` | First-release QoQ log growth, percentage points |
| `selection_matrix.csv` | Binary EN matrix, 180 origins × 585 series |
| `selection_results.json` | Per-origin α, ρ, CV error, training window |
| `selection_matrix_fixedk.csv` | Diagnostic fixed-*k* = 30 path |

## Aggregation and target

Predictors are aggregated with the raw-level bridge in `german_gdp_nowcasting.selection.aggregation`: average the three raw monthly levels, then re-transform (identity for the 398 level-stationary series; quarterly Δln for the 187 growth series). The DFM in notebook 06 stays monthly.

The GDP target uses the first-release vintage rule: both *q* and *q*−1 are read from the earliest vintage in which *q* appears. The vintage matrix begins on 12 May 2005, so 1991Q2–2005Q1 enter training at May 2005 values. Evaluation quarters (2011Q1–2025Q4) are genuine first releases.

## References

Bai & Ng (2008); Bańbura, Giannone & Reichlin (2013); Croushore & Stark (2001); De Mol, Giannone & Reichlin (2008); Friedman, Hastie & Tibshirani (2010); Giannone, Reichlin & Small (2008); Lenza & Primiceri (2022); Zou & Hastie (2005).


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

# --- Portable repository setup ---
_repo = next(
    (base for base in (Path.cwd(), *Path.cwd().parents)
     if (base / "src" / "german_gdp_nowcasting").is_dir()),
    None,
)
if _repo is None:
    raise RuntimeError("Could not locate src/german_gdp_nowcasting.")
_src = _repo / "src"
if str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

from german_gdp_nowcasting.config import paths as _tp
from german_gdp_nowcasting.selection.core_utils import (
    DEFAULT_ALPHAS,
    DEFAULT_L1_RATIOS,
    build_coverage_mask,
    extract_gdp_target,
    load_monthly_panel,
    load_pub_lag_map,
    load_trafo_map,
    make_monthly_forecast_origins,
    monthly_to_quarterly,
    parse_gdp_realtime,
    save_selection_outputs,
)
from german_gdp_nowcasting.selection.elastic_net_selection import (
    covid_sample_weights,
    run_expanding_selection,
)

# Thesis specification (override only for diagnostics)
IMPUTER_STRATEGY = "iterative"   # thesis setting (MICE). "mean" is a faster diagnostic option only.
TSTAT_PREFILTER = True      # Bai & Ng (2008) marginal t-stat pre-screen
TSTAT_THRESHOLD = 1.65      # |t| threshold (one-sided 5%)
MAX_SELECTED = 60           # thesis cap; binds from 2020Q2 onward
N_LAGS = 0                  # distributed lags per series (0 = off, correct default at quarterly freq)
# Lenza & Primiceri (2022): downweight COVID quarters inside ElasticNetCV (row-scaling in ts_elastic_net).
# EN_SAMPLE_WEIGHT is built in §2 after y_quarterly is loaded.
USE_COVID_SAMPLE_WEIGHT = True
COVID_WEIGHT_START = "2020Q2"
COVID_WEIGHT_END = "2021Q1"
COVID_WEIGHT_VALUE = 0.25
EN_SAMPLE_WEIGHT = None  # set in §2
DATA_PATH = _tp.DATASET_XLSX
X_PATH = _tp.PANEL_TRANSFORMED_CSV
DICT_PATH = _tp.DATA_DICT_ENRICHED_CSV
PUB_LAG_PATH = _tp.PUB_LAG_CSV
GDP_TARGET_PATH = _tp.GDP_TARGET_CSV
OUT_DIR = _tp.OUT_INDICATOR_SELECTION
# Elastic Net full expanding-window run (writes selection_matrix.csv + selection_results.json).
# If both files already exist under OUT_DIR, they are loaded and the expensive loop is skipped
# unless FORCE_RERUN_EN_SELECTION = True.
RUN_FULL_SELECTION = True
FORCE_RERUN_EN_SELECTION = False  # True → refit even when CSV/JSON already exist
# Fixed-k = 30 Bai & Ng (2008) targeted-predictor path (diagnostic).
# Block-balanced k=20 (one series per category) is built in run_blockbalanced_benchmark.py, not here.
RUN_FIXEDK = True
FORCE_RERUN_FIXEDK = False  # True → refit even when fixed-k CSV/JSON already exist
FORECAST_START = '2011-01'
FORECAST_END = '2025-12'
MIN_COVERAGE = 0.30       # series must have ≥30% non-NaN in training window
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Repo    : {_tp.REPO_ROOT}')
print(f'Dataset : {_tp.DATASET_XLSX}')
print(f'Data    : {_tp.DATA}')
print(f'Outputs : {_tp.OUTPUTS}')
print(f'Forecast window: {FORECAST_START} → {FORECAST_END}')


## 1. Load inputs and construct the first-release target

Loads the transformed 585-series panel, the enriched dictionary, and the `GDP_realtime` vintage matrix.

`extract_gdp_target` builds quarter-on-quarter log growth from the **first-release** vintage: for each quarter *q*, the leftmost non-missing vintage column supplies both *q* and *q*−1. The vintage matrix starts on 12 May 2005; earlier calendar quarters are May 2005 back-history, not 1990s flash estimates. From 2005Q2, and throughout evaluation, the rule is the genuine flash (Croushore & Stark 2001).


In [ ]:
required_paths = [DATA_PATH, X_PATH, DICT_PATH]
missing_paths = [path for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f'Missing required input files: {missing_paths}')
X_monthly   = load_monthly_panel(X_PATH)
trafo_map   = load_trafo_map(DICT_PATH)
y_quarterly = extract_gdp_target(DATA_PATH)
pub_lag_map = load_pub_lag_map(PUB_LAG_PATH)
# Vintage-matrix span (ifoCAST): columns are publication dates. Growth for each
# quarter uses the leftmost column where that quarter is non-missing. If the
# sheet only lists vintages from (e.g.) 2005 onward, pre-2005 calendar quarters
# still have entries in column 0 — those are back-history levels as of that
# vintage, not the statistical office's original flash from the 1990s.
_gdp_raw = pd.read_excel(DATA_PATH, sheet_name='GDP_realtime', header=None)
_vintages = parse_gdp_realtime(_gdp_raw)
print(
    f'GDP vintage matrix      : {_vintages.shape[0]} quarters × {_vintages.shape[1]} vintages; '
    f'columns { _vintages.columns.min().date() } → { _vintages.columns.max().date() }'
)
_row91 = _vintages.loc['1991Q2']
_first_v91 = _row91[_row91.notna()].index[0]
print(
    f'Example 1991Q2          : first non-NaN column (used for growth) = {_first_v91.date()}'
)
y_quarterly.to_csv(GDP_TARGET_PATH, header=True)
print(f'Monthly predictor panel : {X_monthly.shape[0]} months × {X_monthly.shape[1]} series')
print(f'Predictor range         : {X_monthly.index.min().date()} → {X_monthly.index.max().date()}')
print(f'GDP target              : {len(y_quarterly)} quarters')
print(f'GDP target range        : {y_quarterly.index.min()} → {y_quarterly.index.max()}')
print(f'GDP target saved        : {GDP_TARGET_PATH}')
print(f'Publication lag map     : {len(pub_lag_map)} series  |  lag range {int(pub_lag_map.min())}–{int(pub_lag_map.max())} months')
if set(X_monthly.columns) != set(trafo_map.index):
    raise ValueError('Predictor columns and data dictionary ids are not identical.')
missing_share = X_monthly.isna().mean().describe(percentiles=[0.1, 0.5, 0.9])
print('\nMissing-share distribution across monthly predictors:')
print(missing_share.round(3))


## 2. Quarterly aggregation and coverage mask

The elastic net matches the quarterly target. Aggregation is the raw-level bridge: quarterly **mean** of the three raw monthly levels, then re-transform (`trafo = 0` kept in levels; otherwise quarterly Δln). `run_expanding_selection` recomputes this at each origin.

**Coverage.** A series is eligible if at least 30% of monthly observations in the training window are non-missing (the coverage-eligible panel of the thesis, roughly 580 series per origin).

**Missing values after aggregation.** Iterative imputation (MICE) and standardisation are fitted once per training window, outside the CV loop. This is the thesis setting; `IMPUTER_STRATEGY = "mean"` is only a speed option.

Publication lags are **not** applied here. Part I scores association with completed-quarter growth. The ragged edge is handled in notebook 06.


In [3]:
forecast_origins = make_monthly_forecast_origins(FORECAST_START, FORECAST_END)
X_quarterly = monthly_to_quarterly(X_monthly, trafo_map)
coverage_mask = build_coverage_mask(
    X_monthly,
    forecast_origins,
    min_coverage=MIN_COVERAGE,
)
# series_per_origin[t] = number of series that pass the coverage filter at origin t
series_per_origin = coverage_mask.sum(axis=1)
_n_origins = len(forecast_origins)
print(f'Forecast origins     : {forecast_origins[0]} → {forecast_origins[-1]}  ({_n_origins} monthly origins)')
print(f'Quarterly predictors : {X_quarterly.shape[0]} quarters × {X_quarterly.shape[1]} series')
print()
print(f'Coverage filter (≥{MIN_COVERAGE:.0%} non-NaN observations in training window):')
print(f'  Eligible series at first origin ({forecast_origins[ 0]}) : {series_per_origin.iloc[ 0]} / {X_monthly.shape[1]}')
print(f'  Eligible series at last  origin ({forecast_origins[-1]}) : {series_per_origin.iloc[-1]} / {X_monthly.shape[1]}')
print(f'  Min across all {_n_origins} origins : {series_per_origin.min()}   Max : {series_per_origin.max()}')
print()
print('Residual NaN handling: IterativeImputer (MICE) fitted once per training window (thesis setting).')
# Elastic Net CV observation weights (Lenza & Primiceri 2022) — passed to run_expanding_selection
if USE_COVID_SAMPLE_WEIGHT:
    EN_SAMPLE_WEIGHT = covid_sample_weights(
        y_quarterly,
        start=COVID_WEIGHT_START,
        end=COVID_WEIGHT_END,
        weight=COVID_WEIGHT_VALUE,
    )
    _n_down = int((EN_SAMPLE_WEIGHT < 1.0).sum())
    print()
    print('Elastic Net sample_weight (COVID downweight):')
    print(f'  Quarters {COVID_WEIGHT_START}–{COVID_WEIGHT_END}: weight = {COVID_WEIGHT_VALUE}; else 1.0')
    print(f'  Quarters with w < 1: {_n_down}  |  min weight = {EN_SAMPLE_WEIGHT.min():.3f}')
else:
    EN_SAMPLE_WEIGHT = None
    print()
    print('Elastic Net sample_weight: None (uniform weights in CV)')


Forecast origins     : 2011-01 → 2025-12  (180 monthly origins)
Quarterly predictors : 140 quarters × 585 series

Coverage filter (≥30% non-NaN observations in training window):
  Eligible series at first origin (2011-01) : 567 / 585
  Eligible series at last  origin (2025-12) : 585 / 585
  Min across all 180 origins : 567   Max : 585

Residual NaN handling: IterativeImputer (MICE) fitted once per training window (thesis setting).

Elastic Net sample_weight (COVID downweight):
  Quarters 2020Q2–2021Q1: weight = 0.25; else 1.0
  Quarters with w < 1: 4  |  min weight = 0.250


## 3. Smoke test (optional)

Set `SMOKE_TEST = True` to run the first three origins (2011Q1). They share the 2010Q4 training window, so the net should be fitted once and reused. Skip for a normal run (`SMOKE_TEST = False`).


In [ ]:
SMOKE_TEST = False
import os
if SMOKE_TEST:
    # First 3 monthly origins share the same training window (training end = 2010Q4)
    # so the Elastic Net is fitted once and cached for origins 2 and 3.
    smoke_origins = forecast_origins[:3]
    smoke_mask = coverage_mask.loc[[str(o) for o in smoke_origins]]
    os.environ["THESIS_GRID_NJOBS"] = "3"
    # Match Section 3 (full run): same CV depth and hyperparameter grids. A
    # smaller n_splits / narrower l1_ratio / shorter alpha range skews CV
    # toward very sparse LASSO solutions and is not representative.
    smoke_matrix, smoke_results = run_expanding_selection(
        X_monthly=X_monthly,
        y_quarterly=y_quarterly,
        trafo_map=trafo_map,
        forecast_origins=smoke_origins,
        coverage_mask=smoke_mask,
        train_start_quarter='1991Q1',
        min_selected=1,
        max_selected=MAX_SELECTED,
        n_splits=5,
        l1_ratios=DEFAULT_L1_RATIOS,
        alphas=DEFAULT_ALPHAS,
        imputer_strategy=IMPUTER_STRATEGY,
        tstat_prefilter=TSTAT_PREFILTER,
        tstat_threshold=TSTAT_THRESHOLD,
        n_lags=N_LAGS,
        sample_weight=EN_SAMPLE_WEIGHT,
    )
    # Use the first origin for the detailed report (others are identical — cached)
    origin_key = str(smoke_origins[0])
    res = smoke_results[origin_key]
    # Null-model benchmark: MSE of predicting the training mean for every quarter
    train_end = pd.Period(res['train_end'], freq='Q')
    y_train = y_quarterly.loc[
        pd.Period('1991Q1', freq='Q'):train_end
    ].dropna()
    null_mse = float(y_train.var())
    r2_cv = 1.0 - res['cv_mse'] / null_mse
    print(f"=== Smoke-test origin: {origin_key} ===")
    print(f"  Training window : {res['train_start']} → {res['train_end']}  ({res['n_train_quarters']} quarters)")
    print(
        f"  Candidates      : {res['n_candidate_variables']} / {X_monthly.shape[1]} "
        "series (≥30% monthly coverage)"
    )
    print(f"  Selected        : {res['n_selected_variables']} indicators")
    print(f"  Best alpha      : {res['alpha']:.4f}   l1_ratio: {res['l1_ratio']}")
    print()
    print(f"  CV MSE (Elastic Net)  : {res['cv_mse']:.4f} pp²    RMSE: {res['cv_mse']**0.5:.4f} pp")
    print(f"  Null model MSE (mean) : {null_mse:.4f} pp²    Std(y): {null_mse**0.5:.4f} pp")
    print(
        f"  Approx R²_cv          : {r2_cv:.3f}  "
        f"(1 − CV MSE / Var(y) over full training window; not a foldwise OOS R²)"
    )
    print()
    print(f"  All {res['n_selected_variables']} selected indicators:")
    for var in res['selected_variables']:
        print(f"    {var}")
    print()
    print(f"  Cache check: origins 2011-02 and 2011-03 reuse the same fit:")
    for ok in [str(o) for o in smoke_origins]:
        n = smoke_results[ok]['n_selected_variables']
        end = smoke_results[ok]['train_end']
        print(f"    {ok}  →  train_end={end}  n_selected={n}")
    assert smoke_matrix.shape == (3, X_monthly.shape[1]), "Unexpected selection matrix shape."
    assert smoke_matrix.values.min() >= 0 and smoke_matrix.values.max() <= 1, "Values must be 0/1."
    print()
    print('Smoke test passed.')
else:
    print('Smoke test skipped.')


## 4. Expanding-window elastic net

One `ElasticNetCV` fit per completed quarter (~60 fits for 180 monthly origins). A series is selected if its coefficient is non-zero at the CV-chosen (α, ρ); if more than 60 are non-zero, the 60 largest in absolute value are kept.

COVID quarters 2020Q2–2021Q1 are down-weighted to 0.25 so four extreme observations cannot dominate the path (Lenza & Primiceri 2022).

If `selection_matrix.csv` already exists, it is loaded. Set `FORCE_RERUN_EN_SELECTION = True` to refit. After loading, the next cell prints the set-size range. On the thesis specification that range is 12–60. A maximum above 60 means the CSV predates the cap — cite the thesis tables, not that printout.


In [ ]:
import json
MATRIX_CSV = _tp.SELECTION_MATRIX_CSV
RESULTS_JSON = _tp.SELECTION_RESULTS_JSON
_cache_ok = MATRIX_CSV.exists() and RESULTS_JSON.exists()
if _cache_ok and not FORCE_RERUN_EN_SELECTION:
    selection_matrix = pd.read_csv(MATRIX_CSV, index_col='forecast_origin').astype(int)
    with RESULTS_JSON.open(encoding='utf-8') as f:
        selection_results = json.load(f)
    n_selected = selection_matrix.sum(axis=1)
    print('Existing Elastic Net outputs found — loading from disk (skipping run_expanding_selection).')
    print(f'  {MATRIX_CSV}')
    print(f'  {RESULTS_JSON}')
    print('\nSelected indicators per monthly origin:')
    print(n_selected.describe().round(1))
    print('\nSample (first 6 origins):')
    print(n_selected.head(6).to_string())
    n_sel = selection_matrix.sum(axis=1)
    print(f'EN set size: min={int(n_sel.min())}, median={n_sel.median():.1f}, max={int(n_sel.max())}, mean={n_sel.mean():.1f}')
    if n_sel.max() > MAX_SELECTED:
        print('WARNING: this CSV exceeds the thesis cap of 60. Cite the thesis tables, not this printout.')
    print('\nTo refit from scratch, set FORCE_RERUN_EN_SELECTION = True in the setup cell.')
elif RUN_FULL_SELECTION or FORCE_RERUN_EN_SELECTION:
    selection_matrix, selection_results = run_expanding_selection(
        X_monthly=X_monthly,
        y_quarterly=y_quarterly,
        trafo_map=trafo_map,
        forecast_origins=forecast_origins,
        coverage_mask=coverage_mask,
        train_start_quarter='1991Q1',
        min_selected=1,
        max_selected=MAX_SELECTED,
        n_splits=5,
        l1_ratios=DEFAULT_L1_RATIOS,
        alphas=DEFAULT_ALPHAS,
        imputer_strategy=IMPUTER_STRATEGY,
        tstat_prefilter=TSTAT_PREFILTER,
        tstat_threshold=TSTAT_THRESHOLD,
        n_lags=N_LAGS,
        sample_weight=EN_SAMPLE_WEIGHT,
    )
    save_selection_outputs(
        OUT_DIR,
        selection_matrix=selection_matrix,
        selection_results=selection_results,
    )
    n_selected = selection_matrix.sum(axis=1)
    print('Selection outputs saved:')
    print(f'  {MATRIX_CSV}')
    print(f'  {RESULTS_JSON}')
    print('\nSelected indicators per monthly origin:')
    print(n_selected.describe().round(1))
    print('\nSample (first 6 origins):')
    print(n_selected.head(6).to_string())
else:
    raise FileNotFoundError(
        f'Missing selection outputs ({MATRIX_CSV.name} / {RESULTS_JSON.name}). '
        'Set RUN_FULL_SELECTION = True in the setup cell to fit, or restore the files.'
    )


## 5. Fixed-*k* = 30 diagnostic (not the block-balanced set)

This is the Bai & Ng (2008) targeted-predictor path: traverse the elastic-net path at a fixed mixing weight and stop at the first step with *k* = 30 active coefficients. It is a **dimension diagnostic** for Part I. It is **not** the thesis block-balanced set.

The block-balanced set re-ranks the elastic-net coefficients so that every eligible economic category contributes at least one series, and keeps exactly 20. That set is built in `scripts/pipelines/dfm/run_blockbalanced_benchmark.py` and enters Part II as DFM-block-balanced.

The path here does not apply COVID weights, so the comparison isolates adaptive versus fixed dimension.


In [ ]:
import json
from german_gdp_nowcasting.selection.elastic_net_selection import run_expanding_selection_fixedk

FIXEDK_K = 30
FIXEDK_L2 = 0.25
FIXEDK_MATRIX_PATH = _tp.FIXEDK_MATRIX_CSV
FIXEDK_RESULTS_PATH = _tp.FIXEDK_RESULTS_JSON
_fk_cache_ok = FIXEDK_MATRIX_PATH.exists() and FIXEDK_RESULTS_PATH.exists()
if _fk_cache_ok and not FORCE_RERUN_FIXEDK:
    fixedk_matrix = pd.read_csv(FIXEDK_MATRIX_PATH, index_col='forecast_origin').astype(int)
    with FIXEDK_RESULTS_PATH.open(encoding='utf-8') as f:
        fixedk_results = json.load(f)
    n_fixedk = fixedk_matrix.sum(axis=1)
    print('Existing fixed-k outputs found — loading from disk (skipping run_expanding_selection_fixedk).')
    print(f'  {FIXEDK_MATRIX_PATH}')
    print(f'  {FIXEDK_RESULTS_PATH}')
    print('\nSelected indicators per monthly origin (fixed-k):')
    print(n_fixedk.describe().round(1))
    print('\nSample (first 6 origins):')
    print(n_fixedk.head(6).to_string())
    print('\nTo refit from scratch, set FORCE_RERUN_FIXEDK = True in the setup cell.')
elif RUN_FIXEDK or FORCE_RERUN_FIXEDK:
    print(f'Running fixed-k Bai-Ng baseline: k={FIXEDK_K}, l2={FIXEDK_L2} ...')
    fixedk_matrix, fixedk_results = run_expanding_selection_fixedk(
        X_monthly=X_monthly,
        y_quarterly=y_quarterly,
        trafo_map=trafo_map,
        forecast_origins=forecast_origins,
        coverage_mask=coverage_mask,
        train_start_quarter='1991Q1',
        k=FIXEDK_K,
        l2_penalty=FIXEDK_L2,
        imputer_strategy=IMPUTER_STRATEGY,
    )
    fixedk_matrix.to_csv(FIXEDK_MATRIX_PATH)
    with FIXEDK_RESULTS_PATH.open('w', encoding='utf-8') as f:
        json.dump(fixedk_results, f, indent=2)
    n_fixedk = fixedk_matrix.sum(axis=1)
    print('fixed-k (k=30) selection outputs saved:')
    print(f'  {FIXEDK_MATRIX_PATH}')
    print(f'  {FIXEDK_RESULTS_PATH}')
    print('\nSelected indicators per monthly origin (fixed-k):')
    print(n_fixedk.describe().round(1))
    print('\nSample (first 6 origins):')
    print(n_fixedk.head(6).to_string())
else:
    raise FileNotFoundError(
        f'Missing fixed-k outputs ({FIXEDK_MATRIX_PATH.name} / {FIXEDK_RESULTS_PATH.name}). '
        'Set RUN_FIXEDK = True in the setup cell to fit, or restore the files.'
    )


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
# Compare EN (CV-selected) vs. fixed-k selection counts per origin
if RUN_FIXEDK or FIXEDK_MATRIX_PATH.exists():
    n_en = selection_matrix.sum(axis=1)
    n_fk = fixedk_matrix.reindex(n_en.index).sum(axis=1)
    _x = pd.to_datetime(n_en.index)
    fig, ax = plt.subplots(figsize=(11, 3.5))
    ax.plot(_x, n_en.values, label='EN (CV-selected alpha)', color='steelblue', lw=1.4)
    ax.axhline(FIXEDK_K, color='tomato', ls='--', lw=1.4, label=f'fixed-k = {FIXEDK_K}')
    ax.set_xlabel('Forecast origin (month)')
    ax.set_ylabel('# selected indicators')
    ax.set_title('Elastic net (CV, capped at 60) vs. fixed-k = 30 path — selected series per origin')
    ax.legend()
    ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    ax.xaxis.set_major_locator(mdates.YearLocator(1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
    plt.tight_layout()
    plt.show()
    # Overlap between EN and fixed-k per origin (Jaccard)
    sel_en = selection_matrix.astype(bool)
    sel_fk = fixedk_matrix.astype(bool)
    intersection = (sel_en & sel_fk).sum(axis=1)
    union = (sel_en | sel_fk).sum(axis=1)
    jaccard = intersection / union.replace(0, np.nan)
    _no = len(selection_matrix)
    print(f'Jaccard similarity (EN vs. fixed-k) across {_no} origins:')
    print(jaccard.describe().round(3))


## 6. What the elastic net recovers

On the thesis specification the pattern is category-level, not series-level:

- Hard real-activity series (production, turnover, orders, trade, construction) are 29% of the panel but 70.6–79.9% of EN selected mass across regimes. Surveys are 67% of the panel and 18.0–23.4% of EN mass.
- Lag-0 series are 70% of the panel; the elastic net places 22.4% of its mass on them.
- Only two series are selected at every origin: retail trade turnover excluding vehicles, and accommodation and food services turnover.
- Mean Jaccard overlap with the nineteen ifoCAST indicators is 0.11. Four of those nineteen — all forward-looking survey balances — are never selected.

A stored print of twelve always-selected series is from an earlier **uncapped** run. After loading a capped matrix, the count at frequency 1 should be 2. The DFM-EN input is the origin-by-origin `en_only` matrix, not a frequency-thresholded subset.


In [ ]:
from german_gdp_nowcasting.selection.selection_postprocessing import compute_selection_stability

data_dict = pd.read_csv(_tp.DATA_DICT_ENRICHED_CSV)
stability = compute_selection_stability(selection_matrix, data_dict=data_dict)
print(f'Series ever selected (selection_freq > 0)  : {(stability.selection_freq > 0).sum()} / {len(stability)}')
print(f'Series always selected (selection_freq = 1): {(stability.selection_freq == 1).sum()}')
print()
print('Top 30 most persistently selected indicators:')
cols_show = ['selection_freq', 'n_selected']
if 'category' in stability.columns:
    cols_show.append('category')
print(stability.head(30)[cols_show].to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
_n_origins = len(selection_matrix)
# — Panel 1: selection frequency distribution
ax = axes[0]
ax.hist(stability['selection_freq'], bins=30, color='steelblue', edgecolor='white')
ax.set_xlabel(f'Selection frequency (fraction of {_n_origins} origins)')
ax.set_ylabel('Number of series')
ax.set_title('Distribution of selection frequency')
# — Panel 2: selected count per origin over time
ax = axes[1]
n_sel = selection_matrix.sum(axis=1)
ax.plot(range(len(n_sel)), n_sel.values, color='steelblue', lw=1.4)
ax.axhline(n_sel.mean(), color='tomato', ls='--', lw=1.2, label=f'Mean = {n_sel.mean():.1f}')
# Mark quarter boundaries (every 3rd origin)
q_ticks = list(range(0, len(n_sel), 3))
ax.set_xticks(q_ticks[::4])
ax.set_xticklabels(
    [selection_matrix.index[i] for i in q_ticks[::4]],
    rotation=45, ha='right', fontsize=7
)
ax.set_ylabel('# selected indicators')
ax.set_title('Selected count per monthly origin')
ax.legend(fontsize=8)
# — Panel 3: top-10 categories by total selection count
ax = axes[2]
if 'category' in stability.columns:
    cat_counts = stability.groupby('category')['n_selected'].sum().sort_values(ascending=True).tail(10)
    cat_counts.plot.barh(ax=ax, color='steelblue', edgecolor='white')
    ax.set_xlabel('Total selections (across all origins)')
    ax.set_title('Top categories by total selections')
else:
    ax.text(0.5, 0.5, 'Category labels not available', ha='center', va='center',
            transform=ax.transAxes)
plt.tight_layout()
plt.show()
# Top-30 heatmap: origins × persistently selected series
top30_ids = stability.head(30).index.tolist()
heatmap_data = selection_matrix[top30_ids].astype(float)
fig2, ax2 = plt.subplots(figsize=(16, 5))
im = ax2.imshow(heatmap_data.T, aspect='auto', cmap='Blues', vmin=0, vmax=1, interpolation='none')
ax2.set_yticks(range(len(top30_ids)))
ax2.set_yticklabels(top30_ids, fontsize=7)
# x-axis: show every 12th origin (annual)
xtick_pos = list(range(0, len(selection_matrix), 12))
ax2.set_xticks(xtick_pos)
ax2.set_xticklabels(
    [selection_matrix.index[i] for i in xtick_pos],
    rotation=45, ha='right', fontsize=7
)
ax2.set_xlabel('Forecast origin')
ax2.set_title('Selection heatmap — top 30 most persistently selected series (blue = selected)')
plt.colorbar(im, ax=ax2, fraction=0.02, pad=0.01)
plt.tight_layout()
plt.show()